# Practical 2 - What Is Hidden in a Learned Representation?

**[Advanced Data Science and Machine Learning for Health Research](https://erasmussummerprogramme.nl/summer-programme-courses/advanced-data-science-and-machine-learning-for-health-research)**

---

## The question of this practical

> When an unsupervised model compresses biomedical images,
> **what information ends up in the latent representation — and what else is hidden there?**

## Learning objectives

After this notebook you should be able to:

1. describe **encoder-decoder** architectures and the role of the **bottleneck**;
2. interpret a **reconstruction loss** and per-observation reconstruction error;
3. **extract latent vectors** and treat them as a derived feature matrix;
4. visualise representations with **PCA**;
5. **compare** PCA of the original pixels with PCA of the latent space;
6. perform **latent interpolation** and explain what it does not prove;
7. use learned features for **downstream prediction**;
8. detect **technical / acquisition information** encoded in a latent space;
9. distinguish a **representation** from a **biological explanation**.

## CORE vs ADDITIONAL

> **CORE — everyone should complete**
>
> Run the guided cells, look at the figures, and discuss the questions with your neighbour.
>
> 1. Reconstruction quality
> 2. Latent PCA
> 3. Simulated site / acquisition effect
> 4. Latent interpolation
>
> **ADDITIONAL — optional coding**
>
> Exercises 1–4. These are extra practice if you have time and want to try Python. They are
> not required, and the scientific message of the practical does not depend on writing code.

In a 75–90 minute session, finish the CORE path carefully. Additional coding is only for
groups who want extra practice.

## How this notebook is organised

| Section label | What it asks of you |
|---|---|
| **Concept** | Short theory. Read it. |
| **Health-research perspective** | Why this matters clinically or epidemiologically. |
| **Think before running** | Predict the outcome with your neighbour *before* executing the next cell. |
| **Interpretation** | Think about the scientific questions and discuss them with your neighbour. |
| **Additional exercise** | Optional Python. Skip unless you want extra practice. |

Whenever this notebook asks a question, think it through and discuss it with the person next
to you. You do not need to write answers down.

The notebook is guided: you run prepared cells and discuss. Coding is extra.

## Computational profile

* Runs end to end on a **free Colab CPU runtime**; GPU is used if present but never required.
* One download: **~36 MB**.
* One autoencoder training run: **~2-4 minutes on CPU** (weights are cached, so re-running does not retrain).
* Every other step (PCA, logistic regression, interpolation) takes **seconds**.
* Peak RAM below ~2.5 GB.
* Expected duration: **80-85 minutes**.

---

## Dataset and reproducibility

| Item | Value |
|---|---|
| **Dataset** | BloodMNIST, 28x28 version (MedMNIST v2 collection) |
| **Biomedical modality** | Light microscopy of peripheral blood smears (individual cells, stained) |
| **Source (download)** | Zenodo record [10519652](https://zenodo.org/records/10519652), file `bloodmnist.npz` (35.5 MB), official MedMNIST distribution |
| **Underlying clinical data** | 17,092 images of individual normal blood cells from 3 sources at the Core Laboratory of the Hospital Clinic of Barcelona, acquired with a CellaVision DM96 analyser; donors were free of infection, haematologic or oncologic disease and of pharmacologic treatment at the time of collection (Acevedo et al., 2020) |
| **Labels** | 8 cell types: basophil, eosinophil, erythroblast, immature granulocytes, lymphocyte, monocyte, neutrophil, platelet |
| **Official splits** | 11,959 train / 1,712 validation / 3,421 test (7:1:2 split by the MedMNIST authors) |
| **Sample size used here** | Class-balanced subsets with a fixed seed: **~5,600 train** (700 per class) and **~1,600 test** (200 per class) |
| **Preprocessing by MedMNIST authors** | Source images 3x360x363 centre-cropped to 3x200x200, resized to 3x28x28, stored as `uint8` |
| **Preprocessing in this notebook** | Scale to [0, 1], keep RGB channels, no augmentation. Channel order is (H, W, C) in the file and is transposed to (C, H, W) for PyTorch |
| **License / usage** | MedMNIST data: **CC BY 4.0**; MedMNIST code Apache-2.0; source dataset released CC BY 4.0. **Not for clinical use.** |

### Citations to use if you reuse this data

```
Yang, J., Shi, R., Wei, D., Liu, Z., Zhao, L., Ke, B., Pfister, H., Ni, B. (2023).
MedMNIST v2 - A large-scale lightweight benchmark for 2D and 3D biomedical image
classification. Scientific Data, 10(1), 41.

Yang, J., Shi, R., Ni, B. (2021). MedMNIST Classification Decathlon: A Lightweight
AutoML Benchmark for Medical Image Analysis. IEEE ISBI 2021, 191-195.

Acevedo, A., Merino, A., Alferez, S., Molina, A., Boldu, L., Rodellar, J. (2020).
A dataset of microscopic peripheral blood cell images for development of automatic
recognition systems. Data in Brief, 30, 105474.
```


> We use BloodMNIST because its small image size makes representation-learning experiments possible on CPU while retaining biologically meaningful visual heterogeneity.

> ### Honesty statement about the data
>
> There is **no real patient-level metadata** in this notebook: no age, sex, diagnosis, treatment,
> hospital or scanner variables. The labels are morphological cell types, not patient diagnoses,
> and each image is a single cell rather than a patient.
>
> In Part 7 we **create two artificial study sites** by applying an image transformation we choose
> ourselves. "Site A" and "Site B" are **simulated**, exist only inside this notebook, and must
> never be described as real centres. Everything we conclude about site effects is a statement
> about the simulation, used to reason about what real batch effects would do.

---

# 0. Setup

One cell, no installations required in a standard Colab runtime, no Drive mount, no credentials.

In [ ]:
# =====================================================================
# SETUP CELL - run this first
# =====================================================================
import hashlib
import os
import random
import sys
import time
import urllib.request
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                            confusion_matrix, roc_auc_score, silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# ---------- reproducibility ----------
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
RNG = np.random.default_rng(SEED)

# ---------- device ----------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))

DATA_DIR, WEIGHT_DIR = "medmnist_data", "cached_weights"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(WEIGHT_DIR, exist_ok=True)

# ---------- instructor weights (panic button for class) ----------
# Class default: TRAIN_* = False loads committed checkpoints from ./weights/ or GitHub.
# Set TRAIN_* = True only if you want to reproduce training (~1-4 min on CPU).
WEIGHTS_REPO = "https://github.com/roshchupkin/Advanced_ML_2026/raw/main/weights"
INSTRUCTOR_WEIGHT_DIR = "weights"
os.makedirs(INSTRUCTOR_WEIGHT_DIR, exist_ok=True)


def resolve_weight(filename):
    """Return a local path to `filename`, downloading from GitHub if needed."""
    local = os.path.join(INSTRUCTOR_WEIGHT_DIR, filename)
    if os.path.exists(local):
        return local
    url = f"{WEIGHTS_REPO}/{filename}"
    print(f"instructor weight not found locally - downloading {filename} ...")
    download(url, local)
    return local


def load_or_train(model, filename, train_flag, fit_fn, label):
    """Load instructor weights unless train_flag is True."""
    cache_path = os.path.join(WEIGHT_DIR, filename)
    if train_flag:
        print(f"TRAIN flag is True - training {label} ...")
        history = fit_fn()
        torch.save(model.state_dict(), cache_path)
        print(f"saved trained weights to {cache_path}")
        return history
    path = resolve_weight(filename)
    try:
        state = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    print(f"loaded instructor weights from {path}  (set TRAIN flag to True to retrain)")
    return None


def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def download(url, dest, expected_md5=None):
    if os.path.exists(dest):
        if expected_md5 is None or md5sum(dest) == expected_md5:
            print(f"already present: {dest} ({os.path.getsize(dest)/1e6:.1f} MB)")
            return dest
        print("existing file failed the MD5 check - downloading again")

    def hook(blocks, block_size, total):
        if total > 0:
            print(f"\rdownloading {os.path.basename(dest)}: "
                  f"{min(100.0, 100.0*blocks*block_size/total):5.1f}%", end="")

    t0 = time.time()
    urllib.request.urlretrieve(url, dest, reporthook=hook)
    print(f"\rdownloaded {os.path.basename(dest)}: "
          f"{os.path.getsize(dest)/1e6:.1f} MB in {time.time()-t0:.0f} s")
    if expected_md5 is not None:
        got = md5sum(dest)
        print("MD5 verified." if got == expected_md5
              else f"WARNING: MD5 mismatch (expected {expected_md5}, got {got}).")
    return dest


@contextmanager
def timed(label):
    t0 = time.time()
    yield
    print(f"[{label}] took {time.time() - t0:.1f} s")


def report_environment():
    print("python      :", sys.version.split()[0])
    print("torch       :", torch.__version__)
    print("device      :", DEVICE.type.upper(),
          f"({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")
    print("cpu threads :", torch.get_num_threads())
    try:
        import psutil
        vm = psutil.virtual_memory()
        print("RAM         :", f"{vm.total/1e9:.1f} GB total, {vm.available/1e9:.1f} GB free")
    except Exception:
        pass


plt.rcParams.update({"figure.dpi": 110, "axes.grid": False,
                     "image.interpolation": "nearest", "font.size": 9})
TRAIN_AUTOENCODER = False   # set True to retrain (~2-4 min CPU)

report_environment()
print(f"TRAIN_AUTOENCODER={TRAIN_AUTOENCODER}")
print("\nSetup complete.")

In [ ]:
# =====================================================================
# Download BloodMNIST (28x28) from the official Zenodo record
# =====================================================================
BLOOD_URL = "https://zenodo.org/records/10519652/files/bloodmnist.npz?download=1"
BLOOD_MD5 = "7053d0359d879ad8a5505303e11de1dc"
npz_path = os.path.join(DATA_DIR, "bloodmnist.npz")

# Alternative if Zenodo is unreachable:  !pip install medmnist
#   from medmnist import BloodMNIST; BloodMNIST(split="train", download=True)
download(BLOOD_URL, npz_path, BLOOD_MD5)

blob = np.load(npz_path)
print("\narrays inside the npz:", sorted(blob.files))

CLASS_NAMES = ["basophil", "eosinophil", "erythroblast", "immature granulocyte",
               "lymphocyte", "monocyte", "neutrophil", "platelet"]

X_train_all = blob["train_images"]                       # (N, 28, 28, 3) uint8
y_train_all = blob["train_labels"].ravel().astype(np.int64)
X_test_all = blob["test_images"]
y_test_all = blob["test_labels"].ravel().astype(np.int64)

print(f"\ntrain images {X_train_all.shape} dtype={X_train_all.dtype} "
      f"range=[{X_train_all.min()}, {X_train_all.max()}]")
print(f"{'class':24s} {'train':>7s} {'test':>7s}")
for c, name in enumerate(CLASS_NAMES):
    print(f"{c} {name:22s} {(y_train_all == c).sum():>7d} {(y_test_all == c).sum():>7d}")
print("\nThe source classes are imbalanced; we will draw balanced subsets for teaching clarity.")

In [ ]:
def stratified_subset(y, n_per_class, rng):
    # Indices of a class-balanced random subset; fixed generator => reproducible.
    picked = []
    for c in np.unique(y):
        candidates = np.flatnonzero(y == c)
        picked.append(rng.choice(candidates, size=min(n_per_class, len(candidates)),
                                 replace=False))
    out = np.concatenate(picked)
    rng.shuffle(out)
    return out


def to_tensor(x_uint8):
    # (N, 28, 28, 3) uint8  ->  (N, 3, 28, 28) float32 in [0, 1]
    t = torch.from_numpy(np.ascontiguousarray(x_uint8)).float().div_(255.0)
    return t.permute(0, 3, 1, 2).contiguous()


def to_image(x_chw):
    # (3, 28, 28) tensor  ->  (28, 28, 3) numpy array for plotting
    return np.clip(x_chw.detach().cpu().numpy().transpose(1, 2, 0), 0, 1)


tr_idx = stratified_subset(y_train_all, 700, RNG)
te_idx = stratified_subset(y_test_all, 200, RNG)

X_tr_u8, y_tr = X_train_all[tr_idx], y_train_all[tr_idx]
X_te_u8, y_te = X_test_all[te_idx], y_test_all[te_idx]
X_tr, X_te = to_tensor(X_tr_u8), to_tensor(X_te_u8)

print(f"train tensor {tuple(X_tr.shape)}  ({X_tr.element_size()*X_tr.nelement()/1e6:.1f} MB)")
print(f"test  tensor {tuple(X_te.shape)}  ({X_te.element_size()*X_te.nelement()/1e6:.1f} MB)")
print("class counts (train):", np.bincount(y_tr).tolist())

fig, axes = plt.subplots(2, 8, figsize=(11, 3.2))
for c in range(8):
    ids = np.flatnonzero(y_tr == c)[:2]
    for row, i in enumerate(ids):
        axes[row, c].imshow(X_tr_u8[i])
        axes[row, c].axis("off")
        if row == 0:
            axes[row, c].set_title(CLASS_NAMES[c][:12], fontsize=7)
fig.suptitle("BloodMNIST 28x28 - two examples per cell type", fontsize=10)
plt.tight_layout(); plt.show()

---

# Part 1 - Conceptual introduction

### Concept

```
   x            Encoder            z             Decoder          x_hat
28x28x3   ->   f_theta(x)   ->   R^32     ->    g_phi(z)    ->   28x28x3
 2352                          bottleneck                         2352
 numbers                       32 numbers                        numbers
```

An autoencoder is trained to reproduce its own input while passing the information through a
narrow bottleneck. With mean squared error the objective is

$$ \mathcal{L}(\theta,\phi) \;=\; \frac{1}{N}\sum_{n=1}^{N} \left\| x_n - g_\phi\!\left(f_\theta(x_n)\right) \right\|_2^2 . $$

* **Compression**: 2352 numbers must pass through 32. The model must decide what to keep.
* **Representation learning**: the encoder is trained *without labels*. Nothing in the loss mentions cell type.
* **The reconstruction objective is a proxy**: it rewards whatever explains the most pixel variance, which is dominated by size, overall colour, brightness and coarse shape.
* **Nonlinearity**: with linear layers and MSE, an autoencoder spans the same subspace as PCA. Convolutions plus ReLU let it learn curved manifolds that PCA cannot represent.

| | PCA | Autoencoder |
|---|---|---|
| Mapping | linear, orthogonal | nonlinear, learned |
| Objective | maximise explained variance | minimise reconstruction error |
| Solution | unique, closed form | many local optima, depends on seed/architecture |
| Components | ordered and individually interpretable | unordered, entangled, not individually interpretable |
| Inverse | exact linear projection | learned decoder |

### Health-research perspective

Latent vectors are increasingly used as **derived phenotypes**: inputs to clustering ("disease
subtypes"), to risk models, or as covariates in association analyses. Whatever is *not* encoded
cannot be recovered downstream, and whatever *is* encoded - including scanner, staining batch and
site - will propagate into every downstream analysis.

### Think before running

> **If the reconstructed image looks excellent, does this imply that the latent representation
> contains clinically useful information?**

Think about your answer in one sentence and discuss it with your neighbour. You will be asked to revisit it in Part 3.

---

# Part 2 - A small convolutional autoencoder

### Concept - the architecture we will use

```
Encoder                                     Decoder
Conv 3x3, 3->16, stride 2   -> 16x14x14     Linear 32 -> 32*7*7      -> 32x7x7
Conv 3x3, 16->32, stride 2  -> 32x7x7       ConvTranspose 32->16, s2 -> 16x14x14
Flatten                     -> 1568         ConvTranspose 16->3,  s2 -> 3x28x28
Linear 1568 -> 32           -> z (32)       Sigmoid (outputs in [0,1])
```

Roughly 115k parameters. This is **demonstration-scale training**: a few minutes on a CPU with
5,600 images and 20 epochs. A research-grade autoencoder for this data would use more data, more
capacity, augmentation, a validation-based stopping rule and a careful search over the latent
dimension. Our aim is to be able to *inspect* every part of it.

### Think before running

1. The bottleneck has 32 dimensions for 2352 input numbers, a 73-fold compression. Which
   properties of a blood cell image do you expect to survive: overall colour, cell size, nucleus
   shape, fine chromatin texture, the exact position of granules?
2. Will the reconstruction loss keep falling if we train longer? Would that be good?

In [ ]:
class ConvAutoencoder(nn.Module):
    # Small convolutional autoencoder for 3x28x28 images.
    def __init__(self, latent_dim=32, width=16):
        super().__init__()
        self.latent_dim = latent_dim
        self.c2 = 2 * width
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, width, 3, stride=2, padding=1), nn.ReLU(),        # -> width x 14 x 14
            nn.Conv2d(width, self.c2, 3, stride=2, padding=1), nn.ReLU(),  # -> 2*width x 7 x 7
        )
        self.encoder_fc = nn.Linear(self.c2 * 7 * 7, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, self.c2 * 7 * 7)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(self.c2, width, 4, stride=2, padding=1), nn.ReLU(),  # -> 14x14
            nn.ConvTranspose2d(width, 3, 4, stride=2, padding=1), nn.Sigmoid(),    # -> 28x28
        )

    def encode(self, x):
        return self.encoder_fc(self.encoder_conv(x).flatten(1))

    def decode(self, z):
        h = self.decoder_fc(z).view(-1, self.c2, 7, 7)
        return self.decoder_conv(h)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


autoencoder = ConvAutoencoder(latent_dim=32).to(DEVICE)
n_params = sum(p.numel() for p in autoencoder.parameters())
print(autoencoder)
print(f"\nparameters: {n_params:,}")
with torch.no_grad():
    x_probe = X_te[:4].to(DEVICE)
    x_hat, z_probe = autoencoder(x_probe)
print(f"\ninput  {tuple(x_probe.shape)}  ->  latent {tuple(z_probe.shape)}  ->  "
      f"reconstruction {tuple(x_hat.shape)}")
print(f"compression: {int(np.prod(x_probe.shape[1:]))} numbers -> {autoencoder.latent_dim} "
      f"({np.prod(x_probe.shape[1:]) / autoencoder.latent_dim:.0f}x)")

In [ ]:
def train_autoencoder(model, X, X_val=None, epochs=20, batch=128, lr=2e-3, seed=SEED):
    model.to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"epoch": [], "train_loss": [], "val_loss": []}
    n = len(X)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.from_numpy(np.random.default_rng(seed + epoch).permutation(n))
        running = 0.0
        for start in range(0, n, batch):
            xb = X[order[start:start + batch]].to(DEVICE)
            optimiser.zero_grad(set_to_none=True)
            x_hat, _ = model(xb)
            loss = F.mse_loss(x_hat, xb)
            loss.backward()
            optimiser.step()
            running += loss.item() * len(xb)
        val = float("nan")
        if X_val is not None:
            val = float(np.mean(reconstruction_errors(model, X_val)))
        history["epoch"].append(epoch)
        history["train_loss"].append(running / n)
        history["val_loss"].append(val)
        if epoch % 2 == 0 or epoch == 1:
            print(f"epoch {epoch:3d}/{epochs}   train MSE {running / n:.5f}   test MSE {val:.5f}")
    return history


def reconstruct(model, X, batch=256):
    model.eval()
    outs = []
    with torch.no_grad():
        for start in range(0, len(X), batch):
            x_hat, _ = model(X[start:start + batch].to(DEVICE))
            outs.append(x_hat.cpu())
    return torch.cat(outs)


def encode_all(model, X, batch=256):
    model.eval()
    outs = []
    with torch.no_grad():
        for start in range(0, len(X), batch):
            outs.append(model.encode(X[start:start + batch].to(DEVICE)).cpu())
    return torch.cat(outs).numpy()


def reconstruction_errors(model, X, batch=256):
    # Per-observation mean squared error over all pixels and channels.
    model.eval()
    errs = []
    with torch.no_grad():
        for start in range(0, len(X), batch):
            xb = X[start:start + batch].to(DEVICE)
            x_hat, _ = model(xb)
            errs.append(((x_hat - xb) ** 2).mean(dim=(1, 2, 3)).cpu().numpy())
    return np.concatenate(errs)


with timed("obtain autoencoder weights"):
    history = load_or_train(
        autoencoder, "autoencoder_bloodmnist.pt", TRAIN_AUTOENCODER,
        lambda: train_autoencoder(autoencoder, X_tr, X_te, epochs=20),
        "ConvAutoencoder")

if history is not None:
    plt.figure(figsize=(4.4, 2.6))
    plt.plot(history["epoch"], history["train_loss"], marker="o", label="train MSE")
    plt.plot(history["epoch"], history["val_loss"], marker="s", label="held-out MSE")
    plt.xlabel("epoch"); plt.ylabel("mean squared error"); plt.yscale("log")
    plt.title("Demonstration-scale training"); plt.legend()
    plt.tight_layout(); plt.show()

---

# Part 3 - Reconstruction: what survives the bottleneck?

### Think before running

The next figure shows **original | reconstruction | absolute difference**. Where do you expect the
largest errors: in the flat background, at the cell boundary, or inside the nucleus?

In [ ]:
X_te_hat = reconstruct(autoencoder, X_te)
errors_te = reconstruction_errors(autoencoder, X_te)
print(f"held-out reconstruction MSE: mean {errors_te.mean():.5f}, "
      f"median {np.median(errors_te):.5f}, max {errors_te.max():.5f}")

show = [int(np.flatnonzero(y_te == c)[0]) for c in range(8)]
fig, axes = plt.subplots(3, len(show), figsize=(1.35 * len(show), 4.6))
for col, i in enumerate(show):
    diff = np.abs(to_image(X_te[i]) - to_image(X_te_hat[i])).mean(axis=2)
    axes[0, col].imshow(to_image(X_te[i]))
    axes[0, col].set_title(CLASS_NAMES[y_te[i]][:11], fontsize=7)
    axes[1, col].imshow(to_image(X_te_hat[i]))
    im = axes[2, col].imshow(diff, cmap="inferno", vmin=0, vmax=0.3)
    axes[2, col].set_title(f"MSE {errors_te[i]:.4f}", fontsize=6)
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_ylabel("original")
fig.suptitle("Row 1: original   |   Row 2: reconstruction   |   Row 3: absolute difference",
             fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Best and worst reconstructions, and the error distribution per cell type
order = np.argsort(errors_te)
best, worst = order[:6], order[-6:]

fig, axes = plt.subplots(2, 12, figsize=(13, 2.8))
for col, i in enumerate(best):
    axes[0, col].imshow(to_image(X_te[i])); axes[1, col].imshow(to_image(X_te_hat[i]))
    axes[0, col].set_title(f"best\n{errors_te[i]:.4f}", fontsize=6)
for col, i in enumerate(worst):
    axes[0, col + 6].imshow(to_image(X_te[i])); axes[1, col + 6].imshow(to_image(X_te_hat[i]))
    axes[0, col + 6].set_title(f"worst\n{errors_te[i]:.4f}", fontsize=6)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Six best (left) and six worst (right) reconstructions - top: original, "
             "bottom: reconstruction", fontsize=10)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 2.8))
ax[0].hist(errors_te, bins=40, color="tab:blue")
ax[0].set_xlabel("per-image MSE"); ax[0].set_ylabel("count")
ax[0].set_title("reconstruction error distribution")
ax[1].boxplot([errors_te[y_te == c] for c in range(8)])
ax[1].set_xticks(range(1, 9), [n[:8] for n in CLASS_NAMES], rotation=45, ha="right", fontsize=7)
ax[1].set_ylabel("per-image MSE"); ax[1].set_title("error by cell type")
plt.tight_layout(); plt.show()

print("worst-reconstructed images by class:",
      {CLASS_NAMES[c]: int((y_te[worst] == c).sum()) for c in np.unique(y_te[worst])})

### Interpretation

Think about each question and discuss it with your neighbour.

1. **What information is preserved** in the reconstructions? Be specific: colour, size, boundary,
   nucleus shape, granularity, texture.
2. **What disappears?** Which of the disappearing features would a haematologist use?
3. **Could subtle disease-related information disappear while the reconstruction still looks
   visually good?** Explain using the form of the MSE objective: how many pixels does a small
   nuclear inclusion occupy, and how much does it contribute to the loss?
4. Look at the per-class error distribution. Which cell types are reconstructed worst, and is that
   because they are *rare in the training set*, *visually complex*, or *heterogeneous*?
5. Revisit your answer to the Part 1 question. Would you now change it?

### Health-research perspective

Reconstruction error is a useful **quality-control and outlier statistic**: images that a model
trained on routine data cannot reproduce are often the technically unusual ones (out of focus,
wrong stain, artefacts) - or the rare, genuinely abnormal ones. Distinguishing "technically
weird" from "clinically interesting" cannot be done by the error value alone.

### ADDITIONAL Exercise 1 - reconstruction error as a quality-control statistic (optional coding)

> You do not need to write Python to complete this practical. Skip this cell unless you want extra practice.

Complete the cell below.

1. Split the test set into the 5% of images with the **highest** reconstruction error and the 5%
   with the **lowest** (`np.quantile`).
2. Compare the **class composition** of the two tails (`np.bincount(..., minlength=8)`) and display
   four images from each tail.
3. Now build a deliberately altered copy of the test set - for example
   `X_bright = (X_te * 1.25).clamp(0, 1)` - and compute its reconstruction errors with
   `reconstruction_errors(autoencoder, X_bright)`. Report the mean error before and after.
4. **Discuss with your neighbour:** you propose using reconstruction error as an automated quality-control
   filter in a multicentre study. Which images would be excluded, and what selection bias could
   that introduce?

In [ ]:
# ===== EXERCISE 1 =====
# Available: errors_te, X_te, X_te_u8, y_te, CLASS_NAMES, reconstruction_errors, autoencoder,
#            to_image, np.quantile

# TODO 1: lo, hi = np.quantile(errors_te, [0.05, 0.95])
#         idx_low  = np.flatnonzero(errors_te <= lo)
#         idx_high = np.flatnonzero(errors_te >= hi)

# TODO 2: print np.bincount(y_te[idx_low], minlength=8) and the same for idx_high,
#         then plot 4 images from each group

# TODO 3: X_bright = (X_te * 1.25).clamp(0, 1)
#         err_bright = reconstruction_errors(autoencoder, X_bright)
#         print the mean error before and after

# TODO 4: discuss your interpretation with your neighbour

---

# Part 4 - Extract the latent representations

### Concept

The encoder is now a fixed function $f_\theta: \mathbb{R}^{2352} \to \mathbb{R}^{32}$. Applying it
to every image turns an image dataset into an ordinary tabular dataset:

```
N observations  x  D latent features
```

This matrix is what people mean by a "learned representation", an "embedding", or - in health
research language - a **derived phenotype**: a set of quantitative features computed from raw
measurements, which can then be clustered, associated with outcomes, or fed into a prediction model.

Two properties matter for everything that follows:

* the latent features are **not individually interpretable** and have no natural ordering;
* they were learned **without ever seeing a label**, so they encode whatever explains pixel variance - biology *and* technical variation.

In [ ]:
with timed("encode all images"):
    Z_tr = encode_all(autoencoder, X_tr)
    Z_te = encode_all(autoencoder, X_te)

print(f"Z_tr shape: {Z_tr.shape}   (N observations x D latent features)")
print(f"Z_te shape: {Z_te.shape}")
print(f"one image went from {int(np.prod(X_te.shape[1:]))} pixel values to {Z_te.shape[1]} "
      f"latent values")

latent_table = pd.DataFrame(Z_te[:5, :8],
                            columns=[f"z{j}" for j in range(8)],
                            index=[f"cell {i} ({CLASS_NAMES[y_te[i]][:10]})" for i in range(5)])
print("\nfirst 5 observations, first 8 of 32 latent features:")
print(latent_table.round(2).to_string())

print("\nper-feature summary (test set):")
print(pd.DataFrame({"mean": Z_te.mean(0), "sd": Z_te.std(0)}).describe().round(2).to_string())

---

# Part 5 - PCA of the latent space

### Concept

PCA on the 32 latent dimensions is used here purely as a **2D viewing device**: it finds the
directions of largest variance so we can plot something. Note the layering: a nonlinear
compression (autoencoder) followed by a linear projection (PCA).

### Think before running

Colour by the true cell type. Do you expect the classes to separate, given that the autoencoder
never saw a single label?

In [ ]:
pca_latent = PCA(n_components=min(10, Z_tr.shape[1]), random_state=SEED).fit(Z_tr)
P_lat_te = pca_latent.transform(Z_te)

fig, ax = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.5, 1]})
cmap = plt.get_cmap("tab10")
for c in range(8):
    m = y_te == c
    ax[0].scatter(P_lat_te[m, 0], P_lat_te[m, 1], s=8, alpha=0.65,
                  color=cmap(c), label=CLASS_NAMES[c])
ax[0].set_xlabel(f"PC1 ({100*pca_latent.explained_variance_ratio_[0]:.0f}% of latent variance)")
ax[0].set_ylabel(f"PC2 ({100*pca_latent.explained_variance_ratio_[1]:.0f}%)")
ax[0].set_title("PCA of the 32-dimensional latent space, coloured by cell type")
ax[0].legend(fontsize=7, markerscale=1.5, loc="best")
ax[1].bar(range(1, len(pca_latent.explained_variance_ratio_) + 1),
          100 * pca_latent.explained_variance_ratio_)
ax[1].set_xlabel("principal component"); ax[1].set_ylabel("% of latent variance")
ax[1].set_title("scree plot")
plt.tight_layout(); plt.show()

print(f"first two components carry "
      f"{100*pca_latent.explained_variance_ratio_[:2].sum():.0f}% of the latent variance")
print(f"silhouette score of the true classes in the full latent space: "
      f"{silhouette_score(Z_te, y_te):.3f}  (1 = perfectly separated, 0 = overlapping)")

### Interpretation

Think about each question and discuss it with your neighbour.

1. **Does cell type separate** in this plot? Which classes overlap most?
2. The autoencoder never saw a label. **What does the separation demonstrate?**
3. **What does it NOT demonstrate?** Consider at least: (a) that the representation is *sufficient*
   for diagnosis, (b) that the axes mean anything, (c) that distances are clinically calibrated,
   (d) that structure would replicate in new data from another laboratory.
4. Two thirds of the latent variance sits in the first two components. Is a low-dimensional
   picture therefore a faithful summary of the representation?

---

# Part 6 - PCA before versus after autoencoding

### Concept

We now compare two 32-dimensional representations of the same images:

```
PCA(original pixels)  :  linear projection of 2352 raw values  -> 32 components
PCA(latent features)  :  nonlinear encoder -> 32 latents       -> 32 components (for plotting)
```

Both compress to the same size, so the comparison is fair. The question is whether the nonlinear
encoder organises the data more usefully than a linear projection.

### Think before running

Which representation do you expect to give better *k*-nearest-neighbour classification of cell
type: 32 pixel principal components, or 32 autoencoder latents?

In [ ]:
pixels_tr = X_tr.flatten(1).numpy()
pixels_te = X_te.flatten(1).numpy()

with timed("PCA on raw pixels"):
    pca_pixels = PCA(n_components=32, random_state=SEED).fit(pixels_tr)
P_pix_te = pca_pixels.transform(pixels_te)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for c in range(8):
    m = y_te == c
    ax[0].scatter(P_pix_te[m, 0], P_pix_te[m, 1], s=7, alpha=0.6, color=cmap(c),
                  label=CLASS_NAMES[c])
    ax[1].scatter(P_lat_te[m, 0], P_lat_te[m, 1], s=7, alpha=0.6, color=cmap(c))
ax[0].set_title(f"PCA of raw pixels\nPC1+PC2 = "
                f"{100*pca_pixels.explained_variance_ratio_[:2].sum():.0f}% of pixel variance")
ax[1].set_title("PCA of autoencoder latents")
ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2"); ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2")
ax[0].legend(fontsize=6, markerscale=1.4)
ax[2].plot(np.cumsum(100 * pca_pixels.explained_variance_ratio_), marker="o", ms=3,
           label="pixels")
ax[2].plot(np.cumsum(100 * PCA(n_components=32, random_state=SEED)
                     .fit(Z_tr).explained_variance_ratio_), marker="s", ms=3, label="latents")
ax[2].set_xlabel("number of components"); ax[2].set_ylabel("cumulative % variance explained")
ax[2].set_title("cumulative variance WITHIN each space\n(the two curves are not comparable)")
ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Caution: '% variance explained' refers to the variance of whatever space the PCA was fitted")
print("in. 100% of the latent variance is not 100% of the image information.")

In [ ]:
# A quantitative anchor: k-NN classification in each 32-dimensional space
def knn_accuracy(train_features, test_features, k=10):
    scaler = StandardScaler().fit(train_features)
    model = KNeighborsClassifier(n_neighbors=k).fit(scaler.transform(train_features), y_tr)
    return accuracy_score(y_te, model.predict(scaler.transform(test_features)))


with timed("k-NN comparison"):
    acc_pix_pca = knn_accuracy(pca_pixels.transform(pixels_tr), P_pix_te)
    acc_latent = knn_accuracy(Z_tr, Z_te)

print(f"{'representation':32s} {'dims':>5s} {'10-NN accuracy':>15s}")
print(f"{'PCA of raw pixels':32s} {32:>5d} {acc_pix_pca:>15.3f}")
print(f"{'autoencoder latent space':32s} {Z_tr.shape[1]:>5d} {acc_latent:>15.3f}")
print(f"{'chance level (8 balanced classes)':32s} {'-':>5s} {0.125:>15.3f}")
print(f"\nsilhouette (true classes): pixels-PCA {silhouette_score(P_pix_te, y_te):.3f}, "
      f"latents {silhouette_score(Z_te, y_te):.3f}")

### Interpretation

Think about each point and discuss it with your neighbour:

1. **Linear versus nonlinear.** Did the nonlinear encoder buy anything over PCA at equal
   dimensionality? Quantify with the *k*-NN numbers.
2. **Compression.** Both spaces have 32 dimensions, but PCA components are ordered and the latent
   dimensions are not. Which is easier to report in a paper, and which is easier to *reproduce*?
3. **Interpretability.** You can visualise a pixel principal component as an image (an
   eigen-image). Can you do the equivalent for a latent dimension? (You can decode a perturbed
   latent vector - that is Part 8.)
4. **Clustering.** If a clustering algorithm gave different results in the two spaces, which would
   you believe?
5. **Information loss.** Name one type of information that PCA of pixels keeps and the autoencoder
   probably discards, and one where the opposite holds.

### ADDITIONAL Exercise 2 - the bottleneck controls what is learned (optional coding)

> You do not need to write Python to complete this practical. Skip this cell unless you want extra practice.

Train a **second** autoencoder with `latent_dim=2`, keeping everything else identical but using a
smaller training subset and fewer epochs so that it finishes in about a minute. Then:

1. display original versus reconstruction for the same eight test images and compare with the
   32-dimensional model;
2. scatter the two latent dimensions directly (no PCA needed), coloured by cell type;
3. compute `knn_accuracy` in the 2-dimensional latent space and compare it with the 32-dimensional
   space and with the 32 pixel principal components;
4. discuss with your neighbour the trade-off between compression, reconstruction quality and
   downstream usefulness. Which of the three numbers would you report in a paper?

In [ ]:
# ===== EXERCISE 2 =====
# TODO 1: ae2 = ConvAutoencoder(latent_dim=2).to(DEVICE)
# TODO 2: fit_or_load(ae2, "conv_autoencoder_z2.pt",
#                     lambda: train_autoencoder(ae2, X_tr[:3000], X_te, epochs=10))
# TODO 3: X_te_hat2 = reconstruct(ae2, X_te)   -> plot next to X_te and X_te_hat
# TODO 4: Z2_tr, Z2_te = encode_all(ae2, X_tr), encode_all(ae2, X_te)
#         scatter Z2_te[:, 0] vs Z2_te[:, 1] coloured by y_te
# TODO 5: knn_accuracy(Z2_tr, Z2_te) and compare with acc_latent / acc_pix_pca

---

# Part 7 - Simulated domain shift: two artificial study sites

> ## SIMULATED SITES
>
> "Site A" and "Site B" **do not exist**. We create them by taking the same images and applying a
> mild intensity transformation to a random half of them:
>
> * gamma correction (0.85),
> * small per-channel gain (red and blue slightly increased),
> * a small additive brightness offset,
> * mild Gaussian noise.
>
> This mimics a different staining protocol, illumination setting or camera on an otherwise
> identical population. Any conclusion below is a statement about **this simulation**, not about
> the real BloodMNIST data or the Hospital Clinic of Barcelona.

### Health-research perspective

In multicentre imaging studies, site is almost never randomised with respect to biology: centres
differ in population, referral pattern, protocol *and* device. That makes acquisition a textbook
**confounder** of any image-derived exposure. The vocabulary differs by field - **batch effect**
(omics), **scanner effect** (radiology), **domain shift** (machine learning), **lack of
transportability** (epidemiology) - but the structure is identical.

Crucially, the transformation below is applied to a **random half** of the images, so site is
*independent of cell type by construction*. There is no confounding of the class-site association
here; we are isolating the question "does the representation encode acquisition?".

### Think before running

1. The transformation is barely visible to the eye. Will it move the latent vectors?
2. Could a logistic regression predict *site* from a 32-dimensional latent vector?
3. Do you expect the site effect to appear in the *first* principal components, or in later ones?

In [ ]:
def site_b_effect(x, gamma=0.85, gain=(1.03, 1.0, 1.08), brightness=0.03, noise=0.01, rng=None):
    # SIMULATED acquisition difference applied to (N, 3, 28, 28) tensors in [0, 1].
    rng = np.random.default_rng(0) if rng is None else rng
    out = x.clone().clamp(0, 1).pow(gamma)
    out = out * torch.tensor(gain, dtype=torch.float32).view(1, 3, 1, 1) + brightness
    out = out + torch.from_numpy(rng.normal(0, noise, size=tuple(out.shape)).astype(np.float32))
    return out.clamp(0, 1)


rng_site = np.random.default_rng(42)
site = (rng_site.random(len(X_te)) < 0.5).astype(int)          # 0 = Site A, 1 = Site B
mask_b = torch.from_numpy(site == 1)
X_pool = X_te.clone()
X_pool[mask_b] = site_b_effect(X_te[mask_b], rng=rng_site)
y_pool = y_te.copy()

print(f"Site A: n={(site == 0).sum()},  Site B: n={(site == 1).sum()}")
print(f"class distribution identical by construction: "
      f"A {np.bincount(y_pool[site == 0], minlength=8).tolist()}")
print(f"                                             B "
      f"{np.bincount(y_pool[site == 1], minlength=8).tolist()}")
print(f"mean pixel intensity: A {X_pool[~mask_b].mean():.3f}, B {X_pool[mask_b].mean():.3f} "
      f"(difference {X_pool[mask_b].mean() - X_pool[~mask_b].mean():+.3f})")

ids = np.flatnonzero(site == 0)[:4]
fig, axes = plt.subplots(3, 4, figsize=(7, 5.4))
for col, i in enumerate(ids):
    b_version = site_b_effect(X_te[i:i + 1], rng=np.random.default_rng(1))[0]
    axes[0, col].imshow(to_image(X_te[i])); axes[0, col].set_title("Site A (original)", fontsize=8)
    axes[1, col].imshow(to_image(b_version)); axes[1, col].set_title("Site B (simulated)", fontsize=8)
    axes[2, col].imshow(np.abs(to_image(X_te[i]) - to_image(b_version)).mean(2),
                        cmap="inferno", vmin=0, vmax=0.15)
    axes[2, col].set_title("absolute difference", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("The simulated site effect is subtle by design", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Encode the pooled data again and look at the same latent space twice
Z_pool = encode_all(autoencoder, X_pool)
P_pool = pca_latent.transform(Z_pool)                 # same PCA basis as before

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for c in range(8):
    m = y_pool == c
    ax[0].scatter(P_pool[m, 0], P_pool[m, 1], s=8, alpha=0.6, color=cmap(c), label=CLASS_NAMES[c])
ax[0].set_title("coloured by CELL TYPE (biology)")
ax[0].legend(fontsize=6, markerscale=1.4)
for s, colour, name in [(0, "tab:blue", "Site A"), (1, "tab:orange", "Site B (simulated)")]:
    m = site == s
    ax[1].scatter(P_pool[m, 0], P_pool[m, 1], s=8, alpha=0.6, color=colour, label=name)
ax[1].set_title("coloured by SIMULATED SITE (acquisition)")
ax[1].legend(fontsize=8, markerscale=1.4)
for a in ax:
    a.set_xlabel("PC1 of latent space"); a.set_ylabel("PC2")
plt.tight_layout(); plt.show()

# where does the site effect live? per-component standardised mean difference
diffs = [(P_pool[site == 1, j].mean() - P_pool[site == 0, j].mean()) / P_pool[:, j].std()
         for j in range(P_pool.shape[1])]
plt.figure(figsize=(5.5, 2.4))
plt.bar(range(1, len(diffs) + 1), diffs, color="tab:orange")
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("latent principal component"); plt.ylabel("standardised A vs B difference")
plt.title("Site information is spread across components")
plt.tight_layout(); plt.show()

In [ ]:
# Can a simple model recover the simulated site from the latent representation?
Z_site_tr, Z_site_te, s_tr, s_te = train_test_split(
    Z_pool, site, test_size=0.4, random_state=SEED, stratify=site)

site_clf = LogisticRegression(max_iter=2000)
site_scaler = StandardScaler().fit(Z_site_tr)
site_clf.fit(site_scaler.transform(Z_site_tr), s_tr)
s_prob = site_clf.predict_proba(site_scaler.transform(Z_site_te))[:, 1]

print("PREDICTING SIMULATED SITE FROM THE 32 LATENT FEATURES")
print(f"  accuracy         {accuracy_score(s_te, (s_prob > 0.5).astype(int)):.3f}")
print(f"  balanced accuracy{balanced_accuracy_score(s_te, (s_prob > 0.5).astype(int)):>6.3f}")
print(f"  ROC-AUC          {roc_auc_score(s_te, s_prob):.3f}   (0.5 = no site information)")

# Transportability: train a cell-type model on Site A only, test within and across sites
idx_a = np.flatnonzero(site == 0)
idx_b = np.flatnonzero(site == 1)
a_train, a_test = train_test_split(idx_a, test_size=0.35, random_state=SEED,
                                   stratify=y_pool[idx_a])

cls_scaler = StandardScaler().fit(Z_pool[a_train])
cls_clf = LogisticRegression(max_iter=3000).fit(cls_scaler.transform(Z_pool[a_train]),
                                                y_pool[a_train])
acc_within = balanced_accuracy_score(
    y_pool[a_test], cls_clf.predict(cls_scaler.transform(Z_pool[a_test])))
acc_across = balanced_accuracy_score(
    y_pool[idx_b], cls_clf.predict(cls_scaler.transform(Z_pool[idx_b])))

print("\nCELL-TYPE MODEL TRAINED ON SITE A ONLY (balanced accuracy)")
print(f"  internal validation, Site A held out : {acc_within:.3f}")
print(f"  external validation, Site B          : {acc_across:.3f}")
print(f"  drop                                 : {acc_within - acc_across:+.3f}")

plt.figure(figsize=(4.2, 2.6))
plt.bar(["Site A\n(internal)", "Site B\n(external)"], [acc_within, acc_across],
        color=["tab:blue", "tab:orange"])
plt.axhline(0.125, color="grey", ls="--", lw=0.8)
plt.ylabel("balanced accuracy"); plt.ylim(0, 1.0)
plt.title("Same biology, different acquisition")
plt.tight_layout(); plt.show()

### Interpretation

Think about each question and discuss it with your neighbour.

1. **Has the autoencoder learned biology, acquisition conditions, or both?** Support your answer
   with the two coloured PCA plots *and* the site-prediction AUC.
2. **If site can be predicted with high accuracy from the latent representation, what does that
   mean for multicentre studies?** Work through the consequences for:
   * a clustering analysis that claims to find "patient subtypes";
   * a prediction model trained at one centre and deployed at another;
   * an association study using latent features as exposures or covariates.
3. In our simulation, site is independent of cell type by construction. **What changes if site is
   associated with the outcome** - for example if the second centre serves a sicker population?
   Draw the causal diagram.
4. Which of these would you trust to remove the site signal: per-site standardisation of latent
   features, adding site as a covariate in the downstream model, image-level harmonisation before
   encoding, training the encoder with a site-adversarial loss? What does each break?
5. The cell-type model dropped in balanced accuracy across sites. If instead it had *not* dropped,
   would that prove the representation is site-independent?

### Concept - the same phenomenon, four vocabularies

| Field | Name | Typical remedy |
|---|---|---|
| Genomics / omics | batch effect | ComBat, RUV, include batch in the design |
| Radiology / imaging | scanner or protocol effect | harmonisation, intensity normalisation, multi-vendor training |
| Machine learning | domain shift, covariate shift | domain adaptation, invariance penalties, augmentation |
| Epidemiology | lack of transportability / external validity | multi-site design, external validation, transport formulas |

All four describe the same thing: **a systematic, non-biological source of variation that is
entangled with the signal of interest.** Unsupervised representation learning does not remove it -
it faithfully encodes it.

### ADDITIONAL Exercise 3 - harmonise the simulated site effect (optional coding)

> You do not need to write Python to complete this practical. Skip this cell unless you want extra practice.

The site was highly predictable from the latent vectors. Try the simplest possible harmonisation and
measure what it costs.

1. Standardise the latent features **within each site** (subtract that site's mean and divide by
   that site's standard deviation, separately for each latent dimension). This is the latent-space
   analogue of a batch correction such as ComBat.
2. Refit the site-prediction logistic regression on the harmonised features. Does the ROC-AUC fall
   towards 0.5?
3. Refit the cell-type model on Site A only and evaluate on Site B using harmonised features. Did
   transportability improve?
4. **Discuss with your neighbour:** what did you have to assume in step 1? What happens to this procedure if the
   two sites genuinely differ in class composition - for example if Site B serves a population with
   more immature granulocytes? Relate this to the difference between removing a *nuisance*
   and removing *signal*.

In [ ]:
# ===== EXERCISE 3 =====
# Available: Z_pool, site, y_pool, mask_b, LogisticRegression, StandardScaler,
#            train_test_split, roc_auc_score, balanced_accuracy_score

# TODO 1: Z_harm = Z_pool.copy()
#         for s in (0, 1):
#             m = site == s
#             Z_harm[m] = (Z_pool[m] - Z_pool[m].mean(0)) / (Z_pool[m].std(0) + 1e-8)

# TODO 2: repeat the site-prediction experiment on Z_harm and print the ROC-AUC

# TODO 3: repeat the Site-A-trained cell-type model on Z_harm and report Site A / Site B
#         balanced accuracy

# TODO 4: discuss your interpretation with your neighbour

---

# Part 8 - Latent interpolation

### Concept

Take two images, encode them, and walk along the straight line between their latent vectors:

$$ z(\alpha) \;=\; (1-\alpha)\, z_A \;+\; \alpha\, z_B, \qquad \alpha \in [0, 1], $$

then decode every point. Because the decoder is a smooth function, the output sequence changes
smoothly. Compare this with interpolating the **pixels** directly, which simply cross-fades two
pictures.

### Think before running

Will the decoded midpoint $g_\phi(z(0.5))$ look like (a) a blurred average of the two cells,
(b) a plausible third cell, or (c) something that is not a cell at all?

In [ ]:
i_a = int(np.flatnonzero(y_te == 4)[0])            # a lymphocyte
i_b = int(np.flatnonzero(y_te == 6)[0])            # a neutrophil
alphas = np.linspace(0, 1, 9)

z_a = torch.from_numpy(Z_te[i_a]).float()
z_b = torch.from_numpy(Z_te[i_b]).float()
z_path = torch.stack([(1 - a) * z_a + a * z_b for a in alphas])
with torch.no_grad():
    decoded = autoencoder.decode(z_path.to(DEVICE)).cpu()

pixel_path = torch.stack([(1 - a) * X_te[i_a] + a * X_te[i_b] for a in alphas])

fig, axes = plt.subplots(2, len(alphas), figsize=(1.25 * len(alphas), 3.4))
for col, a in enumerate(alphas):
    axes[0, col].imshow(to_image(pixel_path[col]))
    axes[0, col].set_title(f"a={a:.2f}", fontsize=7)
    axes[1, col].imshow(to_image(decoded[col]))
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_title(f"{CLASS_NAMES[y_te[i_a]][:10]}\na=0.00", fontsize=7)
axes[0, -1].set_title(f"{CLASS_NAMES[y_te[i_b]][:10]}\na=1.00", fontsize=7)
fig.suptitle("Top: interpolation in PIXEL space (cross-fade)   |   "
             "Bottom: interpolation in LATENT space (decoded)", fontsize=9)
plt.tight_layout(); plt.show()

# the same walk between two cells of the SAME type
same = np.flatnonzero(y_te == 6)[:2]
z1, z2 = torch.from_numpy(Z_te[same[0]]).float(), torch.from_numpy(Z_te[same[1]]).float()
with torch.no_grad():
    decoded_same = autoencoder.decode(
        torch.stack([(1 - a) * z1 + a * z2 for a in alphas]).to(DEVICE)).cpu()
fig, axes = plt.subplots(1, len(alphas), figsize=(1.25 * len(alphas), 1.9))
for col in range(len(alphas)):
    axes[col].imshow(to_image(decoded_same[col])); axes[col].axis("off")
fig.suptitle("Latent walk between two neutrophils", fontsize=9)
plt.tight_layout(); plt.show()

### Interpretation

> **Does a smooth visual transition imply a biological trajectory?**

**No**, and it is worth being precise about why:

1. The path is a **straight line in a coordinate system that the model invented**. Nothing
   constrains that line to pass through configurations that occur in real blood.
2. The decoder is **continuous by construction**: it maps nearby vectors to similar images. Smooth
   output is a property of the function class, not evidence about biology.
3. There is **no time** in the model. The interpolation parameter $\alpha$ is not duration,
   progression, dose or severity. Nothing was fitted to longitudinal data.
4. Intermediate images have **no label and no ground truth**. They can be anatomically impossible
   while still looking convincing.
5. The direction $z_B - z_A$ mixes biological and technical variation - including our simulated
   site effect from Part 7.

Now discuss these with your neighbour:

1. What would you need in order to interpret a latent direction as a **disease progression axis**?
   (Think: longitudinal data, an outcome model, identifiability assumptions.)
2. Interpolating between two cells of the same type produced a smooth sequence too. What does that
   tell you about using "smoothness" as evidence of anything?
3. Where could latent interpolation be legitimately useful in health research? Consider data
   augmentation, missing-data imputation, sensitivity analysis and visual communication.

---

# Part 9 - Downstream prediction from learned features

### Concept

```
image  ->  Encoder (trained WITHOUT labels)  ->  z (32 numbers)  ->  logistic regression  ->  cell type
```

This is the standard **linear-probe** protocol: freeze the representation, fit a linear model on
top, and use the accuracy as a measure of how much label-relevant information the representation
contains in a linearly accessible form.

Note the split discipline: the autoencoder saw **only** `X_tr`; the logistic regression is fitted
on `Z_tr` and evaluated on `Z_te`, which comes from the official MedMNIST test split. No test
image influenced either stage.

### Think before running

The encoder never saw a label. How close to a supervised CNN do you expect a linear model on 32
latent features to get, for 8 balanced classes (chance = 12.5%)?

In [ ]:
def evaluate_probe(train_features, test_features, label, max_iter=3000):
    scaler = StandardScaler().fit(train_features)
    clf = LogisticRegression(max_iter=max_iter).fit(scaler.transform(train_features), y_tr)
    proba = clf.predict_proba(scaler.transform(test_features))
    pred = proba.argmax(1)
    metrics = {
        "accuracy": accuracy_score(y_te, pred),
        "balanced accuracy": balanced_accuracy_score(y_te, pred),
        "macro ROC-AUC": roc_auc_score(y_te, proba, multi_class="ovr", average="macro"),
    }
    print(f"{label:34s} " + "  ".join(f"{k} {v:.3f}" for k, v in metrics.items()))
    return metrics, pred


with timed("downstream logistic regressions"):
    m_latent, pred_latent = evaluate_probe(Z_tr, Z_te, "autoencoder latents (32)")
    m_pixpca, _ = evaluate_probe(pca_pixels.transform(pixels_tr), P_pix_te,
                                 "PCA of raw pixels (32)")

print(f"\n{'chance level':34s} accuracy 0.125")

cm = confusion_matrix(y_te, pred_latent, normalize="true")
plt.figure(figsize=(5.2, 4.4))
plt.imshow(cm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(fraction=0.046)
plt.xticks(range(8), [n[:10] for n in CLASS_NAMES], rotation=70, fontsize=7)
plt.yticks(range(8), [n[:10] for n in CLASS_NAMES], fontsize=7)
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("Logistic regression on 32 latent features\n(row-normalised confusion matrix)")
plt.tight_layout(); plt.show()

### Interpretation

Think about each question and discuss it with your neighbour.

1. A linear model on 32 label-free features reaches what accuracy? Compare with the supervised
   benchmark for BloodMNIST (ResNet-18 at 28x28 reaches ~0.958 accuracy in the MedMNIST v2 paper).
   Where does the gap come from?
2. **Does useful representation learning require labels?** Distinguish three regimes: many labels,
   few labels, no labels. In which regime is an unsupervised representation most valuable?
3. Which classes are confused with each other? Are those the pairs a haematologist would also find
   hard, or does the confusion pattern look technical?
4. If you added the *reconstruction error* as a 33rd feature, would you expect the probe to improve?
   What would that feature encode?

### Health-research perspective

The linear-probe protocol is exactly how "foundation model" papers in medical imaging report value:
pretrain without labels on large unlabelled archives, then fit small supervised heads for each
downstream task with limited labels. The economics are attractive because unlabelled images are
abundant while expert labels are not. The risk is that the frozen representation carries site and
protocol information into every downstream task built on top of it.

### ADDITIONAL Exercise 4 - how many labels do you actually need? (optional coding)

> You do not need to write Python to complete this practical. Skip this cell unless you want extra practice.

Learned representations are most valuable when labels are scarce. Build a **label-efficiency
curve**:

1. for `n` in `[50, 100, 250, 500, 1000, len(Z_tr)]`, draw a class-stratified subset of the
   training set;
2. fit the logistic-regression probe on the latent features and on the 32 pixel principal
   components;
3. plot test balanced accuracy against `n` (log x-axis) for both representations;
4. discuss with your neighbour at which sample size the two representations become equivalent, and what that implies for
   a study with 200 expert-labelled images.

In [ ]:
# ===== EXERCISE 4 =====
# hint: reuse stratified_subset(y_tr, n_per_class, RNG) or use
#       sklearn.model_selection.train_test_split(..., stratify=y_tr, train_size=n)

# TODO 1: loop over the sample sizes
# TODO 2: fit LogisticRegression on Z_tr[idx] and on pca_pixels.transform(pixels_tr)[idx]
# TODO 3: evaluate both on the full test set with balanced_accuracy_score
# TODO 4: plot the two curves with plt.semilogx

---

# Connection to the diffusion-model lecture

> **CORE conceptual close** (read only — no training). This bridges to the diffusion lecture without adding compute.

### Concept - why generative models moved into latent space

You have just built the two halves of a **latent generative model**:

```
                    training a generative model in a compressed space
  x  --Encoder-->  z  ~ 32 numbers  --generative model (e.g. diffusion)-->  z'  --Decoder-->  x'
28x28x3                                        operates here                              28x28x3
```

A diffusion model learns to reverse a gradual noising process. Doing that directly in pixel space
is expensive: for a 512x512x3 image the model must denoise 786,432 numbers, hundreds of times per
sample. **Latent diffusion** (Rombach et al., CVPR 2022, the method behind Stable Diffusion) does
exactly what we did here first:

1. train an autoencoder (in practice a VAE with a perceptual and adversarial loss) so that the
   latent space is a faithful, much smaller summary of the image;
2. **freeze** it, and train the diffusion model on latent vectors only;
3. generate a new latent vector by denoising, then decode it into an image.

The saving is the compression factor: a 48x-64x smaller representation means far fewer numbers to
denoise, which is what made high-resolution generative imaging practical.

Two ideas from this notebook transfer directly to that lecture:

* **The bottleneck decides what can be generated.** Anything the encoder discards - fine texture,
  small lesions - cannot be produced by the decoder, no matter how good the diffusion model is.
* **Whatever else is in the latent space is generated too.** If the latent space encodes site,
  staining or scanner, a generative model trained on it will reproduce those factors, and synthetic
  data will inherit the batch structure of the training archive.

Our simple deterministic autoencoder is not a generative model: sampling `z ~ N(0, I)` and decoding
gives noise, because nothing constrained the shape of the latent distribution. That constraint is
what a **variational** autoencoder adds, and it is the reason VAEs (not plain autoencoders) are
used as the first stage of latent diffusion.

*(We do not train a diffusion model here: it would not fit in the compute budget of this practical.)*

---

# Wrap-up

### What we did

```
BloodMNIST images
   -> small convolutional autoencoder (unsupervised, ~2-4 min CPU)
   -> reconstruction quality: what survives 73x compression
   -> latent matrix N x 32: a derived feature representation
   -> PCA of latents vs PCA of pixels: linear vs nonlinear compression
   -> SIMULATED two-site experiment: the latent space encodes acquisition
   -> latent interpolation: smooth is not biological
   -> linear probe: label-free features predict cell type well above chance
   -> bridge to latent diffusion models
```

### Reflection questions

Discuss these with your neighbour, then in the plenary.

1. You are reviewing a paper that clusters autoencoder latents from 4 hospitals and reports three
   "novel patient subtypes". List the three analyses you would require before believing the
   subtypes are biological.
2. Unsupervised learning is often described as "letting the data speak". Given Part 7, what exactly
   is speaking?
3. When would you *prefer* PCA over an autoencoder in a health-research paper? Give two concrete
   situations.
4. A latent representation predicts an outcome well. A collaborator asks which features drive the
   prediction so they can design a biological follow-up experiment. What do you tell them?
5. Reconstruction error, downstream accuracy, site predictability and cluster silhouette are four
   different ways to evaluate a representation. Which would you report as the *primary* metric for
   a representation intended for multicentre epidemiological research, and why?

### Methods you can reuse

| Task | Tool used here |
|---|---|
| unsupervised compression | small convolutional autoencoder, MSE loss |
| per-observation quality metric | reconstruction error |
| representation inspection | PCA + class colouring, silhouette score |
| batch / site diagnostics | logistic regression predicting site from the representation |
| transportability check | train on one site, test on the other |
| representation quality | linear probe with a proper train/test split |

### Key references

* Kingma, D. P., Welling, M. (2014). Auto-Encoding Variational Bayes. *ICLR*.
* Rombach, R. et al. (2022). High-Resolution Image Synthesis with Latent Diffusion Models. *CVPR*.
* Johnson, W. E., Li, C., Rabinovic, A. (2007). Adjusting batch effects in microarray expression data using empirical Bayes methods. *Biostatistics*, 8(1), 118-127.
* Leek, J. T. et al. (2010). Tackling the widespread and critical impact of batch effects in high-throughput data. *Nature Reviews Genetics*, 11, 733-739.
* Glocker, B. et al. (2023). Risk of bias in chest radiography deep learning foundation models. *Radiology: Artificial Intelligence*, 5(6), e230060.
* Chari, T., Pachter, L. (2023). The specious art of single-cell genomics. *PLOS Computational Biology*, 19(8), e1011288. (why 2D embeddings mislead)
* Acevedo, A. et al. (2020). A dataset of microscopic peripheral blood cell images. *Data in Brief*, 30, 105474.
* Yang, J. et al. (2023). MedMNIST v2. *Scientific Data*, 10, 41.

*Model solutions for the additional coding exercises are in `README_Instructor.md`.*